# XBRL financial data API: revenue, earnings and cash flow without tag mapping

Income statement, balance sheet and cash flow from 10-K and 10-Q XBRL, with the same field names for every company. Screen across companies or pull one company's history.

Read it with charts at https://www.quantisnow.com/sec-api/guides/xbrl-financial-data-api. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

XBRL is how companies tag the numbers in their 10-Ks and 10-Qs. The SEC's companyfacts API gives you every tag a company used, under the names that company chose: revenue alone can be `Revenues`, `RevenueFromContractWithCustomerExcludingAssessedTax`, `SalesRevenueNet` and more. The [financials endpoint](https://www.quantisnow.com/sec-api/financial-statements) maps them to one set of fields, fills in fourth quarters from the annual numbers, and keeps one record per company and period.

## One company's quarters

### What were Apple's revenue and net income in each of the last eight quarters?

A quarterly series for a model or a chart, already split into quarters.

```shell
curl -X POST https://www.quantisnow.com/api/v1/financials \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "ticker:AAPL AND fiscalPeriod:(Q1 OR Q2 OR Q3 OR Q4)",
  "from": 0,
  "size": 5,
  "sort": [{"periodEnd": {"order": "desc"}}]
}'
```

Response (real, first 2 of 40):

```json
{
  "total": {
    "value": 40,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0000320193:2026:Q4",
      "cik": "0000320193",
      "ticker": "AAPL",
      "companyName": "Apple Inc.",
      "fiscalYear": 2026,
      "fiscalPeriod": "Q4",
      "periodStart": "2026-06-28",
      "periodEnd": "2026-09-26",
      "filedOn": "2026-11-03",
      "form": "10-K",
      "accessionNo": "0000320193-26-000026",
      "sector": "Technology",
      "currency": "USD",
      "derived": true,
      "incomeStatement": {
        "revenue": 114477000000,
        "costOfRevenue": 59657000000,
        "grossProfit": 54820000000,
        "researchDevelopment": 11730000000,
        "sellingGeneralAdmin": 7459000000,
        "operatingIncome": 35631000000,
        "interestExpense": null,
        "pretaxIncome": 36135000000,
        "incomeTax": 5424000000,
        "netIncome": 30711000000,
        "epsBasic": 2.11,
        "epsDiluted": 2.11,
        "sharesDiluted": null
      },
      "balanceSheet": {
        "cash": 36305000000,
        "shortTermInvestments": 20417000000,
        "receivables": 43817000000,
        "inventory": 14541000000,
        "currentAssets": 173857000000,
        "goodwill": null,
        "totalAssets": 404548000000,
        "currentLiabilities": 169915000000,
        "shortTermDebt": 9906000000,
        "longTermDebt": 68208000000,
        "totalDebt": 78114000000,
        "totalLiabilities": 292455000000,
        "equity": 112093000000
      },
      "cashFlow": {
        "operatingCashFlow": 28101000000,
        "investingCashFlow": 1531000000,
        "financingCashFlow": -32871000000,
        "capex": 2242000000,
        "freeCashFlow": 25859000000,
        "dividendsPaid": 3943000000,
        "shareRepurchases": 24296000000,
        "stockCompensation": 3266000000,
        "depreciation": 3450000000
      }
    },
    {
      "id": "0000320193:2026:Q3",
      "cik": "0000320193",
      "ticker": "AAPL",
      "companyName": "Apple Inc.",
      "fiscalYear": 2026,
      "fiscalPeriod": "Q3",
      "periodStart": "2026-03-29",
      "periodEnd": "2026-06-27",
      "filedOn": "2026-07-31",
      "form": "10-Q",
      "accessionNo": "0000320193-26-000020",
      "sector": "Technology",
      "currency": "USD",
      "derived": false,
      "incomeStatement": {
        "revenue": 109417000000,
        "costOfRevenue": 54647000000,
        "grossProfit": 54770000000,
        "researchDevelopment": 11729000000,
        "sellingGeneralAdmin": 7346000000,
        "operatingIncome": 35695000000,
        "interestExpense": null,
        "pretaxIncome": 36267000000,
        "incomeTax": 6478000000,
        "netIncome": 29789000000,
        "epsBasic": 2.03,
        "epsDiluted": 2.02,
        "sharesDiluted": 14714676000
      },
      "balanceSheet": {
        "cash": 39544000000,
        "shortTermInvestments": 22855000000,
        "receivables": 31398000000,
        "inventory": 11092000000,
        "currentAssets": 149818000000,
        "goodwill": null,
        "totalAssets": 383266000000,
        "currentLiabilities": 149326000000,
        "shortTermDebt": 11007000000,
        "longTermDebt": 71340000000,
        "totalDebt": 82347000000,
        "totalLiabilities": 275746000000,
        "equity": 107520000000
      },
      "cashFlow": {
        "operatingCashFlow": 34369000000,
        "investingCashFlow": -7757000000,
        "financingCashFlow": -32640000000,
        "capex": 2455000000,
        "freeCashFlow": 31914000000,
        "dividendsPaid": 4035000000,
        "shareRepurchases": 25105000000,
        "stockCompensation": 3401000000,
        "depreciation": 3320000000
      }
    }
  ]
}
```

`derived: true` means the quarter wasn't reported on its own and was worked out from the annual figure minus the first three quarters, which is how Q4 is always reported.

## A screen across companies

### Which non-financial companies with over $10 billion in revenue had negative free cash flow last fiscal year?

A screen across every filer in one call, which needs the same field names for every company to work.

```shell
curl -X POST https://www.quantisnow.com/api/v1/financials \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "fiscalPeriod:FY AND fiscalYear:2025 AND incomeStatement.revenue:>10000000000 AND cashFlow.freeCashFlow:<0 AND currency:USD AND NOT sector:Finance",
  "from": 0,
  "size": 5,
  "sort": [{"incomeStatement.revenue": {"order": "desc"}}]
}'
```

Response (real, first 2 of 40):

```json
{
  "total": {
    "value": 40,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0000012927:2025:FY",
      "cik": "0000012927",
      "ticker": "BA",
      "companyName": "Boeing Company",
      "fiscalYear": 2025,
      "fiscalPeriod": "FY",
      "periodStart": "2025-01-01",
      "periodEnd": "2025-12-31",
      "filedOn": "2026-01-30",
      "form": "10-K",
      "accessionNo": "0001628280-26-004357",
      "sector": "Industrials",
      "currency": "USD",
      "derived": false,
      "incomeStatement": {
        "revenue": 89463000000,
        "costOfRevenue": 85174000000,
        "grossProfit": 4289000000,
        "researchDevelopment": 3615000000,
        "sellingGeneralAdmin": 6090000000,
        "operatingIncome": 4281000000,
        "interestExpense": null,
        "pretaxIncome": 2635000000,
        "incomeTax": 397000000,
        "netIncome": 2235000000,
        "epsBasic": 2.49,
        "epsDiluted": 2.48,
        "sharesDiluted": 762300000
      },
      "balanceSheet": {
        "cash": 10921000000,
        "shortTermInvestments": 18479000000,
        "receivables": 2921000000,
        "inventory": null,
        "currentAssets": 128459000000,
        "goodwill": 17275000000,
        "totalAssets": 168235000000,
        "currentLiabilities": 108115000000,
        "shortTermDebt": 8461000000,
        "longTermDebt": 45637000000,
        "totalDebt": 54098000000,
        "totalLiabilities": 162778000000,
        "equity": 5454000000
      },
      "cashFlow": {
        "operatingCashFlow": 1065000000,
        "investingCashFlow": 499000000,
        "financingCashFlow": -3763000000,
        "capex": 2942000000,
        "freeCashFlow": -1877000000,
        "dividendsPaid": 331000000,
        "shareRepurchases": null,
        "stockCompensation": 426000000,
        "depreciation": 1953000000
      }
    },
    {
      "id": "0001001085:2025:FY",
      "cik": "0001001085",
      "ticker": "BN",
      "companyName": "Brookfield Corporation",
      "fiscalYear": 2025,
      "fiscalPeriod": "FY",
      "periodStart": "2025-01-01",
      "periodEnd": "2025-12-31",
      "filedOn": "2026-03-18",
      "form": "40-F",
      "accessionNo": "0001001085-26-000006",
      "sector": "Real Estate",
      "currency": "USD",
      "derived": false,
      "incomeStatement": {
        "revenue": 75100000000,
        "costOfRevenue": 28351000000,
        "grossProfit": 46749000000,
        "researchDevelopment": null,
        "sellingGeneralAdmin": 5131000000,
        "operatingIncome": null,
        "interestExpense": null,
        "pretaxIncome": null,
        "incomeTax": 1135000000,
        "netIncome": 1307000000,
        "epsBasic": 0.51,
        "epsDiluted": 0.49,
        "sharesDiluted": 2367800000
      },
      "balanceSheet": {
        "cash": 16242000000,
        "shortTermInvestments": null,
        "receivables": 22772000000,
        "inventory": 5923000000,
        "currentAssets": null,
        "goodwill": 43355000000,
        "totalAssets": 518971000000,
        "currentLiabilities": null,
        "shortTermDebt": null,
        "longTermDebt": 13800000000,
        "totalDebt": 13800000000,
        "totalLiabilities": 352777000000,
        "equity": 166194000000
      },
      "cashFlow": {
        "operatingCashFlow": 10958000000,
        "investingCashFlow": -31777000000,
        "financingCashFlow": 21687000000,
        "capex": 14752000000,
        "freeCashFlow": -3794000000,
        "dividendsPaid": null,
        "shareRepurchases": null,
        "stockCompensation": null,
        "depreciation": 10379000000
      }
    }
  ]
}
```

## Revenue growth in Python

In [ ]:
import os, requests

BASE = "https://www.quantisnow.com/api/v1"
KEY = os.environ["QUANTISNOW_API_KEY"]

def search(endpoint, query, size=50, sort=None, start=0):
    body = {"query": query, "size": size, "from": start}
    if sort:
        body["sort"] = sort
    res = requests.post(f"{BASE}/{endpoint}", json=body,
                        headers={"Authorization": f"Bearer {KEY}"}, timeout=30)
    res.raise_for_status()
    return res.json()
import pandas as pd

page = search("financials", "ticker:MSFT AND fiscalPeriod:(Q1 OR Q2 OR Q3 OR Q4)",
              sort="periodEnd:desc", size=12)
df = pd.json_normalize(page["data"]).set_index("periodEnd").sort_index()
rev = df["incomeStatement.revenue"]
print((rev / rev.shift(4) - 1).dropna().round(3))  # year-over-year growth by quarter

## In JavaScript

```javascript
const BASE = "https://www.quantisnow.com/api/v1";

async function search(endpoint, query, { size = 50, sort, from = 0 } = {}) {
  const res = await fetch(BASE + "/" + endpoint, {
    method: "POST",
    headers: {
      Authorization: "Bearer " + process.env.QUANTISNOW_API_KEY,
      "Content-Type": "application/json",
    },
    body: JSON.stringify({ query, size, from, sort }),
  });
  if (!res.ok) throw new Error((await res.json()).error.message);
  return res.json();
}

// Annual operating margin for a list of companies
for (const t of ["AAPL", "MSFT", "GOOGL"]) {
  const { data } = await search("financials", "ticker:" + t + " AND fiscalPeriod:FY", { sort: "fiscalYear:desc", size: 1 });
  const s = data[0].incomeStatement;
  console.log(t, data[0].fiscalYear, (s.operatingIncome / s.revenue).toFixed(3));
}
```

For a full walk through loading statements into a DataFrame, see [SEC financial statements in pandas](https://www.quantisnow.com/sec-api/guides/financial-statements-pandas).